# Instruction fine-tuning on two GPUs

Configure the run, inspect the dataset and environment, and launch the standalone Accelerate training script. Training runs in separate processes; the model is saved to the output directory.


## 1. Setup and configuration
Run from the repository or a subdirectory of it. Change the settings below before launching.


In [1]:
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys

project_root = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / "model_training").is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError("Run this notebook from inside the deep_learning repository")

script = project_root / "model_training/post_training/02-sft/train_instruction.py"
data_file = project_root / "datasets/instruction_tuning/instruction-data.json"
output_dir = project_root / "execution_results/qwen_instruction"
model_name = "Qwen/Qwen2.5-3B-Instruct"
num_gpus = 2
epochs = 2
batch_size = 1
accumulation_steps = 4
max_length = 1024
learning_rate = 1e-5
weight_decay = 0.1
print(f"[STAGE] setup | repository: {project_root} | Python: {sys.executable}")
print(f"✓ model: {model_name} | GPUs: {num_gpus} | output: {output_dir}")


[STAGE] setup | repository: /storage/ice1/1/0/vchopra37/projects/learning_env | Python: /home/hice1/vchopra37/scratch/envs/learning_python_env/bin/python
✓ model: Qwen/Qwen2.5-3B-Instruct | GPUs: 2 | output: /storage/ice1/1/0/vchopra37/projects/learning_env/execution_results/qwen_instruction


## 2. Check environment and data
The JSON file must already exist. The original instruction-tuning notebook downloads it. The script uses the first 85% for training.


In [2]:
print("[STAGE] environment and dataset")
missing = [name for name in ("torch", "transformers", "accelerate") if importlib.util.find_spec(name) is None]
if missing:
    raise ModuleNotFoundError(f"Missing packages in the kernel environment: {', '.join(missing)}")
if not script.is_file():
    raise FileNotFoundError(f"Training script not found: {script}")
if not data_file.is_file():
    raise FileNotFoundError(f"Dataset not found: {data_file}; download it using the instruction-tuning notebook")
with data_file.open(encoding="utf-8") as f:
    data = json.load(f)
if not isinstance(data, list) or not data:
    raise ValueError("Dataset must be a non-empty JSON list")
import torch
print(f"✓ examples: {len(data)} | training split: {int(len(data) * 0.85)}")
print(f"✓ visible CUDA GPUs: {torch.cuda.device_count()}")
if torch.cuda.device_count() < num_gpus:
    raise RuntimeError(f"Requested {num_gpus} GPUs but fewer are visible to this kernel")
print(f"✓ nominal effective batch size: {batch_size * num_gpus * accumulation_steps} examples")


[STAGE] environment and dataset


FileNotFoundError: Training script not found: /storage/ice1/1/0/vchopra37/projects/learning_env/model_training/post_training/02-sft/train_instruction.py

## 3. Build the launch command
The process runs with the same Python environment as this notebook. Accelerate starts a separate worker per GPU.


In [ ]:
command = [
    sys.executable, "-m", "accelerate.commands.launch",
    "--multi_gpu", "--num_processes", str(num_gpus), "--mixed_precision", "bf16",
    str(script),
    "--data", str(data_file), "--output", str(output_dir), "--model", model_name,
    "--epochs", str(epochs), "--batch-size", str(batch_size),
    "--accumulation-steps", str(accumulation_steps), "--max-length", str(max_length),
    "--learning-rate", str(learning_rate), "--weight-decay", str(weight_decay),
]
env = os.environ.copy()
env["PYTHONPATH"] = str(project_root) + os.pathsep + env.get("PYTHONPATH", "")
print("[STAGE] launch configuration")
print(f"✓ script: {script} | dataset: {data_file}")
print(f"✓ epochs: {epochs} | batch/GPU: {batch_size} | accumulation: {accumulation_steps}")
print(f"✓ max length: {max_length} | learning rate: {learning_rate}")


## 4. Run training
This cell starts the actual training job and waits for it to finish. A failed worker raises an error in the notebook.


In [ ]:
print("[STAGE] training launch", flush=True)
try:
    subprocess.run(command, cwd=project_root, env=env, check=True)
except subprocess.CalledProcessError as exc:
    print(f"✗ {type(exc).__name__}: training exited with status {exc.returncode}", flush=True)
    raise
print("✓ training process completed", flush=True)


## 5. Inspect saved artifacts
The script saves the model and tokenizer on successful completion.


In [ ]:
print(f"[STAGE] saved artifacts | directory: {output_dir}")
if not output_dir.is_dir():
    raise FileNotFoundError(f"Output directory not found: {output_dir}")
artifacts = sorted(p.name for p in output_dir.iterdir() if p.is_file())
print("✓ files:", artifacts)


## Notes

- The training script uses the existing instruction collator. Its Qwen padding and end-of-message behavior should be verified before relying on a full run.
- The script currently reports the last batch loss each epoch; it does not evaluate validation loss.
- Check the output directory before rerunning if you need to preserve a previous model.
